In [ ]:
import os
import sys
import ast
import json
from unittest.mock import MagicMock
from typing import Dict, Any, List

# Mock Configuration for the Notebook Environment
os.environ["LLM_ENABLED"] = "true"
print("Environment configured for Research & Verification.")

## Track A: Verify Import Dependencies (AST Analysis)

We must ensure that `execution` modules **never** import `llm_integration` or `api`. This prevents "Authority Creep" where the LLM might accidentally influence trading logic. We use Python's Abstract Syntax Tree (AST) to statically analyze code.

In [ ]:
def check_imports(source_code: str, forbidden_modules: List[str]) -> List[str]:
    """Scans source code for forbidden imports using AST."""
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return ["Syntax Error in Source Code"]
        
    violations = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if any(alias.name.startswith(m) for m in forbidden_modules):
                    violations.append(alias.name)
        elif isinstance(node, ast.ImportFrom):
            if node.module and any(node.module.startswith(m) for m in forbidden_modules):
                violations.append(node.module)
    return violations

# --- SIMULATION ---

# 1. Mock Execution Code (Simulating a violation)
bad_execution_code = """
import math
from core.math_engine import calculate_risk
import llm_integration.bridge  # <--- VIOLATION!
"""

# 2. Mock Execution Code (Clean)
good_execution_code = """
import math
from core.math_engine import calculate_risk
# No LLM imports here, pure math.
"""

print("--- Testing Bad Code (Should Fail) ---")
violations = check_imports(bad_execution_code, ["llm_integration", "api"])
if violations:
    print(f"  CRITICAL ARCHITECTURE VIOLATION: Found imports {violations}")
else:
    print("  Code is clean.")

print("\n--- Testing Good Code (Should Pass) ---")
violations = check_imports(good_execution_code, ["llm_integration", "api"])
if violations:
    print(f"  CRITICAL ARCHITECTURE VIOLATION: Found imports {violations}")
else:
    print("  Code is clean.")

## Track A: Verify LLM Return Type Safety

The LLM interface must only return **data** (dictionaries, strings, numbers). It must never return executable objects, functions, or classes that could be instantiated and run by the core system.

In [ ]:
def validate_llm_output(output: Any) -> bool:
    """Ensures LLM output is strictly data (JSON-serializable primitives)."""
    allowed_types = (str, int, float, bool, dict, list, type(None))
    
    if not isinstance(output, allowed_types):
        return False
        
    if isinstance(output, dict):
        # Recursive check for dicts
        return all(validate_llm_output(v) for v in output.values())
    if isinstance(output, list):
        # Recursive check for lists
        return all(validate_llm_output(v) for v in output)
        
    return True

# --- SIMULATION ---

# Mock LLM Response (Safe)
safe_response = {
    "summary": "Strategy performed well.",
    "metrics": {"sharpe": 1.5, "drawdown": -0.1},
    "is_actionable": False
}

# Mock LLM Response (Unsafe - contains a lambda function)
unsafe_response = lambda x: x + 1 

print(f"Safe Response Valid: {validate_llm_output(safe_response)}")
print(f"Unsafe Response Valid: {validate_llm_output(unsafe_response)}")

## Track A: Runtime Kill-Switch

If `LLM_ENABLED` is set to `false` (e.g., during a market panic or system degradation), the system must **not** initialize any LLM components. Trading must continue unaffected.

In [ ]:
def initialize_system(llm_enabled: bool):
    """Simulates system startup sequence."""
    components = ["MarketData", "ExecutionEngine", "RiskManager"]
    
    if llm_enabled:
        print("  LLM Enabled: Initializing Cognitive Layer...")
        components.append("LLMEngine")
        components.append("VectorStore")
        components.append("ExplanationAPI")
    else:
        print("  LLM Disabled: Running in Pure Deterministic Mode.")
        
    return components

# --- SIMULATION ---

print("--- Initializing with LLM_ENABLED=True ---")
active_components_full = initialize_system(True)
print(f"Active Components: {active_components_full}")

print("\n--- Initializing with LLM_ENABLED=False ---")
active_components_safe = initialize_system(False)
print(f"Active Components: {active_components_safe}")

# Verification
assert "ExecutionEngine" in active_components_safe
assert "LLMEngine" not in active_components_safe
print("\n  Kill-Switch Verified: Execution persists, LLM is gone.")

## Track B: Research Acceleration (Safe Read-Only)

Here we demonstrate how to use the LLM to explain backtest results. The flow is:
1.  Run Backtest (Deterministic) -> Get Metrics
2.  Pass Metrics to LLM (Read-Only)
3.  Get Text Explanation (Insight)

This process **never** writes back to the trading config.

In [ ]:
# 1. Mock Backtest Data (From the Deterministic Engine)
backtest_results = {
    "strategy_id": "STRAT_ALPHA_01",
    "period": "2023-Q4",
    "sharpe_ratio": 0.8,
    "max_drawdown": -0.15,
    "win_rate": 0.45,
    "total_trades": 120
}

def mock_llm_explain_backtest(metrics: Dict) -> str:
    """
    Simulates an LLM explaining the metrics.
    In a real scenario, this would call the `llm_integration.api.generate_explanation` endpoint.
    """
    # Logic to generate explanation based on metrics (Simulating LLM inference)
    explanation = f"### Analysis for {metrics['strategy_id']}\n"
    
    if metrics['sharpe_ratio'] < 1.0:
        explanation += "- **Risk-Adjusted Returns**: Suboptimal (Sharpe < 1.0). The strategy is taking too much risk for the return generated.\n"
    
    if metrics['max_drawdown'] < -0.10:
        explanation += f"- **Drawdown Warning**: Max drawdown is {metrics['max_drawdown']:.1%}, which exceeds the typical 10% safety threshold.\n"
        
    if metrics['win_rate'] < 0.50:
        explanation += "- **Win Rate**: Low (45%). This implies a trend-following profile where winners must be significantly larger than losers.\n"
        
    explanation += "\n**Research Recommendation**: Investigate stop-loss parameters to curb drawdown. Do not deploy without optimization."
    return explanation

# 2. Generate Insight
print("--- LLM Research Report ---")
report = mock_llm_explain_backtest(backtest_results)
print(report)

## Track B: Alpha Decay & Regime Analysis

Using the LLM to narrate structural market changes based on HMM (Hidden Markov Model) outputs. This helps researchers understand *why* a strategy might be failing.

In [ ]:
# 1. Mock Regime Data (From Market Intelligence Layer)
regime_data = {
    "current_regime": "High Volatility / Bearish",
    "previous_regime": "Low Volatility / Bullish",
    "transition_confidence": 0.89,
    "alpha_decay_observed": True,
    "volatility_spike": 2.5  # sigma
}

def mock_llm_explain_regime(data: Dict) -> str:
    """
    Simulates LLM narrative generation for regime changes.
    """
    return (
        f"### MARKET REGIME SHIFT DETECTED\n"
        f"**Transition**: {data['previous_regime']}  ️ {data['current_regime']}\n"
        f"**Confidence**: {data['transition_confidence']:.0%}\n"
        f"**Volatility Impact**: {data['volatility_spike']}σ spike detected.\n\n"
        f"**Alpha Status**: {' ️ DECAY DETECTED' if data['alpha_decay_observed'] else 'Stable'}\n"
        "**Insight**: The strategy's mean-reversion logic is likely failing due to the sustained directional volatility. "
        "Consider pausing execution until volatility normalizes."
    )

# 2. Generate Narrative
print("--- Market Intelligence Brief ---")
narrative = mock_llm_explain_regime(regime_data)
print(narrative)